# 第 8 章 · 记忆（ADK：用工具改 state）

`session.state` 是这次会话的临时数据。**改它有三条路**，这是第一条：**在工具里改**。

为什么推荐用工具？因为它把"改状态"这件事**收在一个可测试的函数里**，
而不是散落在 instruction 的自然语言里（`LlmAgent` 自己根本写不了 state）。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

try:
    from google.adk.runners import InMemoryRunner
    from google.genai import types
except ModuleNotFoundError:
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时的 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state

from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext


def log_user_login(tool_context: ToolContext) -> dict:
    """Updates the session state upon a user login event.

    Args:
        tool_context: Automatically provided by ADK, gives access to session state.

    Returns:
        A dictionary confirming the action was successful.
    """
    # ★ 本章新增：tool_context.state 是**可写**的 —— 工具是改 state 的正规入口
    tool_context.state["last_login_user"] = "alice"
    tool_context.state["login_count"] = tool_context.state.get("login_count", 0) + 1
    return {"status": "ok", "user": "alice"}


login_agent = LlmAgent(
    name="LoginAgent",
    model=adk_model(),                      # 【适配】书里没给 model
    instruction="用户说要登录时，调用 log_user_login 工具，然后把返回的 status 复述一遍。",
    tools=[log_user_login],
)

finals, state = run_once(login_agent, "我要登录")
print("agent 答复:", finals[-1][1] if finals else "(无)")
print("\nsession.state =", {k: v for k, v in state.items()})

agent 答复: 登录成功，状态为：ok。

session.state = {'last_login_user': 'alice', 'login_count': 1}
